# ETL
Učitavamo biblioteke i originalne product/review CSV fajlove. Raw podaci ostaju neizmenjeni, a sve transformacije rade se na kopijama.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import hashlib
import html
import json

In [2]:
path_parent=Path.cwd().parent/"data"/"raw"
raw_prods=pd.read_csv(path_parent/'cleaned_makeup_products.csv')
raw_reviews=pd.read_csv(path_parent/'cleaned_makeup_reviews.csv')

C:\Users\USER\AppData\Local\Temp\ipykernel_18644\602964070.py:3: DtypeWarning: Columns (0: uri, 1: comments, 2: locale, 3: location, 4: bottom_line, 5: product_page_id) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_reviews=pd.read_csv(path_parent/'cleaned_makeup_reviews.csv')


## 1. Identitet proizvoda i SKU varijante

Iz URL-a se ukljanja SKU parametar i dobija kanonski proizvod, a `item_id_clean` koristi kao identifikator konkretne SKU varijante.

In [3]:
product_rows = raw_prods.copy()
product_rows['canonical_product_link']=product_rows['product_link'].str.split('?').str[0]
product_rows['sku_from_url']=product_rows['product_link'].str.split('sku=').str[1]
product_rows["item_id_raw"]=pd.to_numeric(product_rows.item_id, errors='coerce').astype('Int64').astype('string')
product_rows['item_id_clean']=[source if pd.isna(source)==False else sku for source,sku in zip(product_rows.item_id_raw,product_rows.sku_from_url) ]

## 2. Mapa izvornih ID-jeva i tabela varijanti

Pravljenje `product_id_map` za povezivanje svih izvornih product redova sa kanonskim proizvodom i zasebnu tabelu `product_variants` sa jednim redom po SKU varijanti.

In [4]:
map_columns = [
    "product_link_id",
    "product_link",
    "canonical_product_link",
    "item_id",
    "item_id_raw",
    "sku_from_url",
    "item_id_clean",
]
product_id_map=product_rows[map_columns].copy()
product_id_map=product_id_map.sort_values(by='product_link_id').reset_index(drop=True)


In [5]:
product_variants=product_rows.groupby(by=['item_id_clean']).agg(
    canonical_product_link=('canonical_product_link','first'),
    product_name=('product_name','first'),
    product_links=('product_link', lambda x: sorted(list(x.unique()))),
    source_product_link_ids=('product_link_id', lambda x: sorted(list(x.unique()))),
    number_of_source_rows=('item_id_clean', 'count')
    ).sort_index(level='item_id_clean').reset_index()


## 3. Čišćenje atributa proizvoda

Čisti opis, izdvaja ili dopunja brend i priprema kategoriju. Izvorne verzije se čuvaju radi provera, a očišćene vrednosti koriste se u kanonskoj tabeli.

### Description

In [6]:
product_rows['description_raw']=product_rows['description'].astype('string')
product_rows['description_clean_candidate']=product_rows.description_raw.map(lambda x:x.split('Summary',1)[1].split('Shipping & Coupon Restrictions',1)[0])
product_rows['description_clean_candidate']=product_rows.description_clean_candidate.map(lambda x:re.sub(r'\s+', ' ', x).strip())
product_rows['description_clean_candidate']=product_rows.description_clean_candidate.str.strip()


### Brand

In [7]:
invalid_brands = ["Ask A Question","Find your shade","Write A Review"]
brand_overrides = {"Born This Way Soft Matte Foundation": "Too Faced","ORIGINAL Liquid Mineral Concealer": "bareMinerals"}

#nadji pozicije product_name u description
product_rows["product_name_position"] = product_rows.apply(
    lambda x: (x["description_raw"].find(x["product_name"]) if pd.notna(x["description_raw"])and pd.notna(x["product_name"]) else -1),
    axis=1)

# tekst do prvog pojavljivanja naziva proizvoda
product_rows["text_before_product_name"] = product_rows.apply(
    lambda x: (x["description_raw"][:x["product_name_position"]]if x["product_name_position"] >= 0 else pd.NA),
    axis=1)

product_rows['brand']=product_rows['brand'].astype('string')
product_rows["brand_clean_candidate"] = product_rows["brand"].astype("string").str.strip().replace(invalid_brands, pd.NA)
product_rows["brand_from_description_candidate"] = product_rows["text_before_product_name"].astype("string").str.rsplit("image", n=1).str[-1].str.replace(r"(?i)^\s*try it\s*","",regex=True).str.strip().replace("", pd.NA)
product_rows['brand_manual_override']=product_rows.product_name.str.strip().map(brand_overrides)
product_rows["brand_final"] = product_rows["brand_clean_candidate"].combine_first(product_rows["brand_from_description_candidate"]).combine_first(product_rows["brand_manual_override"])

from_structured=product_rows.brand_clean_candidate.notna()
from_description=(product_rows.brand_from_description_candidate.notna() & product_rows.brand_clean_candidate.isna()) 
from_manual=product_rows.brand_manual_override.notna()

product_rows["brand_source"] = pd.NA
product_rows.loc[from_structured,"brand_source"]='source'
product_rows.loc[from_description,"brand_source"]='description'
product_rows.loc[from_manual,"brand_source"]='manual_override'

### Category

In [8]:
category_overrides = {
    "https://www.ulta.com/p/halo-sculpt-glow-face-palette-with-vitamin-e-pimprod2042910": "Contouring",
    "https://www.ulta.com/p/mini-cc-cream-with-spf-50-pimprod2013015": "BB & CC Creams",
    'https://www.ulta.com/p/dior-forever-fluid-skin-glow-foundation-pimprod2036824':'Foundation',
    'https://www.ulta.com/p/barepro-24hr-wear-skin-perfecting-matte-liquid-foundation-mineral-spf-20-pimprod2043221':'Foundation',
    'https://www.ulta.com/p/futurist-hydra-rescue-moisturizing-foundation-spf-45-pimprod2013152':"Foundation",
    'https://www.ulta.com/p/trick-treat-cc-active-propolis-color-correcting-cream-with-broad-spectrum-spf-45-pimprod2005371':'Tinted Moisturizer',
    'https://www.ulta.com/p/complexion-rescue-natural-matte-tinted-moisturizer-mineral-spf-30-pimprod2037151':'Tinted Moisturizer',
    'https://www.ulta.com/p/tinted-moisturizer-oil-free-natural-skin-perfector-broad-spectrum-spf-20-pimprod2025045':'Tinted Moisturizer',
    'https://www.ulta.com/p/mini-tinted-moisturizer-natural-skin-perfector-broad-spectrum-spf-30-pimprod2039349':'Tinted Moisturizer',
    'https://www.ulta.com/p/mini-tinted-moisturizer-oil-free-natural-skin-perfector-broad-spectrum-spf-20-pimprod2039444':'Tinted Moisturizer',
    'https://www.ulta.com/p/one-step-correct-brightening-correcting-primer-xlsImpprod2390219':'Face Primer',
    'https://www.ulta.com/p/buttermelt-pressed-powder-blush-pimprod2045333':'Blush',
    'https://www.ulta.com/p/smashbox-x-becca-under-eye-brightening-corrector-pimprod2028907':'Concealer',
     'https://www.ulta.com/p/pro-collagen-cleansing-balm-xlsImpprod18731145':'Makeup Remover',
    'https://www.ulta.com/p/travel-size-pro-collagen-cleansing-balm-pimprod2004741':'Makeup Remover',
    'https://www.ulta.com/p/superfood-aha-glow-cleansing-butter-pimprod2021098':'Makeup Remover',
    'https://www.ulta.com/p/pure-plush-gentle-deep-cleansing-foam-xlsImpprod13481005':'Makeup Remover',
    'https://www.ulta.com/p/take-day-off-charcoal-cleansing-balm-makeup-remover-pimprod2036304':'Makeup Remover'
}

product_rows["category_raw"]=product_rows.category.astype('string')
product_rows["category_clean"]=product_rows[product_rows.category_raw.notna()].category_raw.map(lambda x: re.sub(r'\s+', ' ', x).strip() )
category_summary=product_rows.groupby(by='canonical_product_link').agg(
    number_of_categories=('category_clean', 'nunique'), ##ima i sa null + kat
    categories=('category_clean',lambda x: sorted(x.dropna().unique())),
    number_of_source_rows=('product_link_id','count')
)
product_rows["category_primary"]=product_rows["canonical_product_link"].map(category_summary["categories"].map(lambda categories: (categories[0]if len(categories) == 1 else pd.NA)))
product_rows["category_primary"] =(product_rows["canonical_product_link"].map(category_overrides)).combine_first(product_rows["category_primary"])

overriden_categories = product_rows["canonical_product_link"].isin(category_overrides)
product_rows["category_source"] = 'source'
product_rows.loc[overriden_categories,"category_source"]='manual_override'


### Rešavanje konflikata za različit description zbog sku varijanti po kanonskom proizvodu

In [9]:
#za 5 proizvoda postoje razl opisi po nijansi, pa kanonskom proizvodu dajemo samo zaj deo pre ingredients
#inace opis koji vec postoji

conflict_summary=product_rows.groupby(by='canonical_product_link' ).nunique()[['description_clean_candidate']]
conflict_links=conflict_summary[conflict_summary.gt(1, axis=0).any(axis=1)].index.tolist()
product_rows ["description_before_ingredients"] = product_rows.description_clean_candidate.str.split("Ingredients",n=1,).str[0].str.strip()
product_rows["description_for_canonical"] =product_rows["description_clean_candidate"]
description_conflict_mask = product_rows["canonical_product_link"].isin(conflict_links)
product_rows.loc[description_conflict_mask,"description_for_canonical"] = product_rows.loc[description_conflict_mask,"description_before_ingredients"]

## 4. Kanonska tabela proizvoda

Objedinjuje izvorne redove u products_canonical, sa tačno jednim redom po kanonskom URL-u

In [10]:
products_canonical=product_rows.groupby(by='canonical_product_link').agg(
    product_name=('product_name','first'),
    brand_final=('brand_final', 'first'),
    brand_source=('brand_source',lambda x:sorted(x.unique()) ),
    category_primary=('category_primary', 'first'),
    category_source=('category_source','first'),
    category_all=('category',lambda x:sorted(x.dropna().unique()) ),
    price=('price', 'first'),
    description_clean=('description_for_canonical','first'),
    item_ids=('item_id_clean',lambda x:sorted(x.dropna().unique())),
    variant_count=('item_id_clean','nunique'),
    number_of_source_rows=('product_link_id','count'),

    page_rating=("rating", "first"),
    page_review_count=("num_reviews", "first"),
    average_rating=("average_rating", "first"),
    rating_count=("rating_count", "first"),
    review_count=("review_count", "first"),
    recommended_ratio=("recommended_ratio", "first"),

)
products_canonical = products_canonical.reset_index()

## 5. Dokumenti za recommender

Od naziva, brenda, kategorije i  opisa pravi jedan  `document_text` po proizvodu, bez review polja i brojčanih agregata recenzija proizvoda.

In [11]:
document_columns = [
    "canonical_product_link",
    "product_name",
    "brand_final",
    "category_primary",
    "price",
    "description_clean",
]
product_documents=products_canonical.loc[:,document_columns].copy()
product_documents['document_version']='metadata_v1'
product_documents["document_text"] = (
    "Name: "
    + product_documents["product_name"]
    + "\nBrand: "
    + product_documents["brand_final"]
    + "\nCategory: "
    + product_documents["category_primary"]
    + "\nDescription: "
    + product_documents["description_clean"]
)

## 6. Povezivanje recenzija sa proizvodima

Preko product_link_id mapira review redove na kanonski proizvod. Nepovezane redove izdvaja i ne koristi ih u modelskim tabelama.

In [12]:
reviews = raw_reviews.copy()

#Da li review product_link_id postoji u product_id_map?
unmatched_reviews=reviews[~reviews.product_link_id.isin(product_id_map.product_link_id.values.tolist())]
unmatched_reviews = unmatched_reviews.copy()
unmatched_reviews["unmatched_reason"] = ("product_link_id_not_in_products")

matched_reviews=reviews[reviews.product_link_id.isin(product_id_map.product_link_id.values.tolist())]
matched_reviews_with_product=matched_reviews.merge(product_id_map, on='product_link_id', how='left', validate='many_to_one')

known_product_ids = set(product_id_map["product_link_id"])

## 7. Uklanjanje tehničkih kopija

Uklanja potpuno ponovljene raw redove koji se razlikuju samo po tehničkom  unique_review_id

In [13]:
#tehnicke kopije
technical_duplicate_key=raw_reviews.columns.tolist()
technical_duplicate_key.remove ('unique_review_id')
matched_reviews_deduplicated = matched_reviews_with_product.drop_duplicates(subset=technical_duplicate_key,keep="first").copy()

## 8. Rekonstrukcija logičkih recenzija

Grupiše pure_text, image i video redove prema kanonskom proizvodu i review_id.
Proverava konflikte i rekonstruišemo jednu logičku recenziju sa svim medijima.

In [14]:
logical_review_key = ["canonical_product_link","review_id"]

logical_review_rows =matched_reviews_deduplicated.copy()
logical_review_rows['created_datetime']=pd.to_datetime(logical_review_rows['created_date'], utc=True)
logical_review_rows['image_uri_candidate']=logical_review_rows[logical_review_rows.type.eq('image')].uri
logical_review_rows['video_uri_candidate']=logical_review_rows[logical_review_rows.type.eq('video')].uri


In [15]:
reviews_logical=logical_review_rows.groupby(by=logical_review_key).agg(
    headline=('headline','first'),
    comments=('comments','first'),
    rating=('rating','first'),
    nickname=('nickname','first'),
    created_date_min=('created_datetime','min'),
    created_date_max =('created_datetime','max'),             
    review_effective_date =('created_datetime','max'),
    raw_row_count=('unique_review_id','count'),
    source_unique_review_ids=('unique_review_id',lambda x: sorted(x)),
    source_product_link_ids=('product_link_id',lambda x: sorted(x.unique())),
    item_ids=('item_id_clean',lambda x: sorted(x.unique())),
    types=('type',lambda x: sorted(x.unique())),
    image_uris=('image_uri_candidate',lambda x: sorted(x.dropna().unique())),
    video_uris=('video_uri_candidate',lambda x: sorted(x.dropna().unique())),
)

In [16]:
reviews_logical = reviews_logical.reset_index()
reviews_logical["image_count"] = reviews_logical["image_uris"].str.len()
reviews_logical["video_count"] = reviews_logical["video_uris"].str.len()
reviews_logical["has_image"] = reviews_logical["image_count"].gt(0)
reviews_logical["has_video"] = reviews_logical["video_count"].gt(0)
reviews_logical["has_full_comment"] = reviews_logical["comments"].fillna("").str.strip().ne("")
reviews_logical["has_created_date_conflict"] = reviews_logical["created_date_min"]!= reviews_logical["created_date_max"]

## 9. Isti komentar pod različitim review ID-jevima

Normalizovani komentar koristimo samo za poređenje. Različite `review_id` spajam jedino kada se poklapaju proizvod, neprazan komentar, poznati autor, ocena i naslov.

In [17]:
def normalize_review_text(series):
    return (
        series
        .astype("string")
        .str.lower()
        .str.strip()
        .str.replace(
            r"\s+",
            " ",
            regex=True,
        )
    )

strong_duplicate_key = ["canonical_product_link","normalized_comment","normalized_nickname","rating","headline_match_key",]

reviews_logical["normalized_comment"] = normalize_review_text(reviews_logical["comments"])
reviews_logical["normalized_nickname"] = normalize_review_text(reviews_logical["nickname"])
reviews_logical["normalized_headline"] = normalize_review_text(reviews_logical["headline"])
reviews_logical["headline_match_key"] = (reviews_logical["normalized_headline"].fillna("<MISSING>"))

eligible_for_cross_review_id_merge = (
    reviews_logical["normalized_comment"].notna()
    & reviews_logical["normalized_comment"].ne("")
    & reviews_logical["normalized_nickname"].notna()
    & reviews_logical["normalized_nickname"].ne("")
    & reviews_logical["rating"].notna()
)
eligible_reviews = reviews_logical.loc[eligible_for_cross_review_id_merge].copy()

strong_group_summary = eligible_reviews.groupby(strong_duplicate_key,dropna=False).agg(
        number_of_rows=("review_id","size",),
        number_of_review_ids=("review_id","nunique",),
        review_ids=("review_id",lambda values: sorted(values.dropna().unique().tolist()),),
    ).reset_index()
strong_duplicate_groups = strong_group_summary[strong_group_summary["number_of_review_ids"].gt(1)].copy()
strong_duplicate_groups["strong_duplicate_group_id"] = range(len(strong_duplicate_groups))

strong_duplicate_rows = eligible_reviews.merge(
        strong_duplicate_groups[strong_duplicate_key + ["strong_duplicate_group_id"]],
        on=strong_duplicate_key,
        how="inner",
        validate="many_to_one",
    )

## 10. Izrada kanonske tabele recenzija

Razdvaja redove koji se povezuju kroz review_id od onih koji ostaju pojedinačni, čuva kompletne informacije i sastavlja `reviews_canonical` sa stabilnim ID-jem.


In [18]:
def combine_unique_lists(series):
    combined_values = set()
    for values in series.dropna():
        combined_values.update(values)
    return sorted(combined_values)

keys_to_merge = strong_duplicate_rows[["canonical_product_link","review_id",'strong_duplicate_group_id']].drop_duplicates()
keys_to_merge["is_in_keys_to_merge"] = True
reviews_logical_marked = (reviews_logical.merge(
        keys_to_merge,
        on=["canonical_product_link","review_id"],
        how="left",
        validate="one_to_one",
    )
)

reviews_logical_marked["is_in_keys_to_merge"] = reviews_logical_marked["is_in_keys_to_merge"].fillna(False).astype(bool)
#isti kom, isti komentator , isti proizv, razl review id 
reviews_to_merge = reviews_logical_marked.loc[reviews_logical_marked["is_in_keys_to_merge"]].copy()
reviews_to_keep = reviews_logical_marked.loc[~reviews_logical_marked["is_in_keys_to_merge"]].copy()

In [19]:
reviews_to_merge_sorted = reviews_to_merge.sort_values(["strong_duplicate_group_id", "review_id"])
merged_reviews = reviews_to_merge_sorted.groupby("strong_duplicate_group_id").agg(
    canonical_product_link=("canonical_product_link", "first"),
    source_review_ids=("review_id", lambda x: sorted(x.unique().tolist())),
    duplicate_group_size=("review_id", "nunique"),
    headline=("headline", "first"),
    nickname=("nickname", "first"),
    rating=("rating", "first"),
    comments=("comments", "first"),
    image_uris=("image_uris", combine_unique_lists),
    video_uris=("video_uris", combine_unique_lists),
    source_review_dates=("review_effective_date", lambda x: sorted(x.dropna().unique().tolist())),
    review_effective_date=("review_effective_date", "max"),
    raw_row_count=("raw_row_count", "sum"),
    source_unique_review_ids=("source_unique_review_ids", combine_unique_lists),
    source_product_link_ids=("source_product_link_ids", combine_unique_lists),
    item_ids=("item_ids", combine_unique_lists),
    types=("types", combine_unique_lists),
    created_date_min=("created_date_min", "min"),
    created_date_max=("created_date_max", "max"),
).reset_index()


In [20]:
kept_reviews = reviews_to_keep.copy()
kept_reviews ['source_review_ids']=[[x] for x in kept_reviews.review_id.tolist()] 
kept_reviews ['duplicate_group_size']=1
kept_reviews ['source_review_dates']= kept_reviews.review_effective_date.apply(lambda x: [x] if pd.notna(x) else [])

In [21]:
def make_canonical_review_id(row):
    source_ids = ",".join(
    str(review_id)for review_id in sorted(row["source_review_ids"])
    )

    value_to_hash = (f"{row['canonical_product_link']}|{source_ids}")
    return hashlib.sha256(value_to_hash.encode("utf-8")).hexdigest()
canonical_columns = [
    "canonical_product_link",
    "headline",
    "comments",
    "rating",
    "nickname",
    "image_uris",
    "video_uris",
    "source_review_ids",
    "duplicate_group_size",
    "source_review_dates",
    "review_effective_date",
    "raw_row_count",
    "source_unique_review_ids",
    "source_product_link_ids",
    "item_ids",
    "types",
    "created_date_min",
    "created_date_max",
]

reviews_canonical = pd.concat([kept_reviews[canonical_columns], merged_reviews[canonical_columns]], ignore_index=True)

reviews_canonical["was_cross_review_id_merged"] = reviews_canonical["duplicate_group_size"].gt(1)
reviews_canonical["canonical_review_id"] = (
reviews_canonical.apply(make_canonical_review_id,axis=1))
reviews_canonical['image_count']=reviews_canonical['image_uris'].map(lambda x: len(x))
reviews_canonical['video_count']=reviews_canonical['video_uris'].map(lambda x: len(x))
reviews_canonical['has_image']=reviews_canonical['image_count']>0
reviews_canonical['has_video']=reviews_canonical['video_count']>0
reviews_canonical['has_full_comment']=((reviews_canonical['comments'].str.strip().notna()) &(reviews_canonical['comments'].str.strip().ne('')))
reviews_canonical['has_headline']=((reviews_canonical['headline'].str.strip().notna()) &(reviews_canonical['headline'].str.strip().ne('')))

## 11. Porvera oznake za ponovljene tekstove

Označava iste komentare koji nisu bezbedni za spajanje i tekstove koji se pojavljuju kod više proizvoda. Ovi redovi se ne brišu, nego ostaju dostupni za audit i eksperimente.

Označiti ih da bismo kasnije mogli proveriti da li ponovljeni tekstovi nepravedno utiču na recommender

In [22]:
reviews_canonical["normalized_comment"] = normalize_review_text(reviews_canonical["comments"])
has_comment = reviews_canonical["normalized_comment"].notna() & reviews_canonical["normalized_comment"].ne("")
number_of_products_per_comment = reviews_canonical.groupby("normalized_comment")["canonical_product_link"].transform("nunique")

reviews_canonical["is_same_text_review_candidate"] =has_comment& reviews_canonical.duplicated(subset=["canonical_product_link", "normalized_comment"],keep=False)
reviews_canonical["is_cross_product_reused_text"] = has_comment& number_of_products_per_comment.gt(1)


In [23]:
same_text_review_candidates = reviews_canonical.loc[reviews_canonical["is_same_text_review_candidate"]].groupby(["canonical_product_link", "normalized_comment"]).agg(
    number_of_reviews=("canonical_review_id", "size"),
    canonical_review_ids=("canonical_review_id", lambda x: sorted(x.unique().tolist())),
).reset_index()

cross_product_reused_text = reviews_canonical.loc[reviews_canonical["is_cross_product_reused_text"]].groupby("normalized_comment").agg(
    number_of_reviews=("canonical_review_id", "size"),
    number_of_products=("canonical_product_link", "nunique"),
    canonical_product_links=("canonical_product_link", lambda x: sorted(x.unique().tolist())),
).reset_index()

## 11. Čišćenje teksta recenzija

Čuva raw tekst, dekodira HTML entitete i normalizuje whitespace. Ne uklanja negacije, interpunkciju niti kratke informativne komentare.

In [24]:
html_entity_mask = reviews_canonical["comments"].str.contains(r"&(?:[A-Za-z]+|#\d+|#x[0-9A-Fa-f]+);",regex=True,na=False,)

def clean_review_series(series):
    return series.astype("string").map(html.unescape, na_action="ignore").str.replace(r"\s+", " ", regex=True).str.strip().replace("", pd.NA)

reviews_canonical["headline_raw"] = reviews_canonical["headline"]
reviews_canonical["comments_raw"] = reviews_canonical["comments"]
reviews_canonical["headline_clean"] = clean_review_series(reviews_canonical["headline_raw"])
reviews_canonical["comments_clean"] = clean_review_series(reviews_canonical["comments_raw"])

reviews_canonical["has_full_comment"] = reviews_canonical["comments_clean"].notna()
reviews_canonical["has_headline"] = reviews_canonical["headline_clean"].notna()
reviews_canonical["review_char_length"] = reviews_canonical["comments_clean"].str.len()
reviews_canonical["review_word_count"] = reviews_canonical["comments_clean"].str.split().str.len()

## 12. Tekstualni skup recenzija

U reviews_text uključuje samo kanonske recenzije sa punim komentarom. Naslov koristi kao dopunski tekst, dok headline-only i media-only recenzije ostaju u reviews_canonical.

In [25]:
reviews_text = reviews_canonical.loc[reviews_canonical["has_full_comment"]].copy()
reviews_text["review_text"] = (reviews_text["headline_clean"].fillna("") + " "+ reviews_text["comments_clean"]).str.strip()

## 16. Validacija tabela

Na jednom mestu proverava broj redova, jedinstvenost ključeva, povezanost tabela i uslove koje konačni podaci moraju zadovoljiti.

In [26]:
final_checks = {
    "product_id_map_rows": len(product_id_map) == 1_373,
    "product_variants_rows": len(product_variants) == 1_278,
    "product_variants_unique_item_id": product_variants["item_id_clean"].is_unique,
    "products_canonical_rows": len(products_canonical) == 1_268,
    "products_canonical_unique": products_canonical["canonical_product_link"].is_unique,
    "products_brand_complete": products_canonical["brand_final"].notna().all(),
    "products_category_complete": products_canonical["category_primary"].notna().all(),
    "products_description_complete": products_canonical["description_clean"].notna().all(),
    "unmatched_review_rows": len(unmatched_reviews) == 8_298,
    "reviews_canonical_rows": len(reviews_canonical) == 60_368,
    "reviews_canonical_unique": reviews_canonical["canonical_review_id"].is_unique,
    "reviews_have_known_product": reviews_canonical["canonical_product_link"].isin(products_canonical["canonical_product_link"]).all(),
    "reviews_text_rows": len(reviews_text) == 25_797,
    "reviews_text_unique": reviews_text["canonical_review_id"].is_unique,
    "reviews_text_nonempty": reviews_text["review_text"].str.strip().ne("").all(),
    "documents_unique": product_documents["canonical_product_link"].is_unique,
    "documents_have_known_product": product_documents["canonical_product_link"].isin(products_canonical["canonical_product_link"]).all(),
}

final_check_report = pd.Series(final_checks, name="passed").to_frame()
display(final_check_report)

assert final_check_report["passed"].all(), final_check_report.loc[~final_check_report["passed"]]

,passed
product_id_map_rows,True
product_variants_rows,True
product_variants_unique_item_id,True
products_canonical_rows,True
products_canonical_unique,True
products_brand_complete,True
products_category_complete,True
products_description_complete,True
unmatched_review_rows,True
reviews_canonical_rows,True


## 17. Čuvanje izlaza

Tabele sortira prema  ključevima, liste pretvara u JSON i čuva kao UTF-8 CSV fajlove. 

In [28]:
output_dir = Path.cwd().parent / "data" / "processed"
audit_dir = Path.cwd().parent / "data" / "audit"
output_dir.mkdir(parents=True, exist_ok=True)
audit_dir.mkdir(parents=True, exist_ok=True)

def prepare_for_csv(dataframe, sort_columns):
    result = dataframe.sort_values(sort_columns, kind="stable").reset_index(drop=True).copy()
    for column in result.columns:
        if result[column].map(lambda x: isinstance(x, list)).any():
            result[column] = result[column].map(lambda x: json.dumps(x, ensure_ascii=False, default=str) if isinstance(x, list) else x)
    return result

output_tables = {
    "product_id_map.csv": prepare_for_csv(product_id_map, ["product_link_id"]),
    "product_variants.csv": prepare_for_csv(product_variants, ["item_id_clean"]),
    "products_canonical.csv": prepare_for_csv(products_canonical, ["canonical_product_link"]),
    "unmatched_reviews.csv": prepare_for_csv(unmatched_reviews, ["unique_review_id"]),
    "reviews_canonical.csv": prepare_for_csv(reviews_canonical, ["canonical_review_id"]),
    "reviews_text.csv": prepare_for_csv(reviews_text, ["canonical_review_id"]),
    "product_documents.csv": prepare_for_csv(product_documents, ["canonical_product_link"]),
}
audit_tables = {
    "same_text_review_candidates.csv": prepare_for_csv(same_text_review_candidates, ["canonical_product_link", "normalized_comment"]),
    "cross_product_reused_text.csv": prepare_for_csv(cross_product_reused_text, ["normalized_comment"]),
}

for filename, dataframe in output_tables.items():
    dataframe.to_csv(output_dir / filename, index=False, encoding="utf-8")
for filename, dataframe in audit_tables.items():
    dataframe.to_csv(audit_dir / filename, index=False, encoding="utf-8")

assert all((output_dir / filename).exists() for filename in output_tables)
assert all((audit_dir / filename).exists() for filename in audit_tables)
display(pd.Series(sorted(output_tables), name="processed_file").to_frame())
display(pd.Series(sorted(audit_tables), name="audit_file").to_frame())


,processed_file
0,product_documents.csv
1,product_id_map.csv
2,product_variants.csv
3,products_canonical.csv
4,reviews_canonical.csv
5,reviews_text.csv
6,unmatched_reviews.csv


,audit_file
0,cross_product_reused_text.csv
1,same_text_review_candidates.csv


In [29]:
quality_report = {
    "etl_version": "1.0.0",
    "generated_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "stable_metrics": {
        "raw_product_rows": len(raw_prods),
        "raw_review_rows": len(raw_reviews),
        "product_id_map_rows": len(product_id_map),
        "product_variant_rows": len(product_variants),
        "canonical_product_rows": len(products_canonical),
        "unmatched_review_rows": len(unmatched_reviews),
        "matched_technical_duplicates_removed": len(matched_reviews_with_product) - len(matched_reviews_deduplicated),
        "canonical_review_rows": len(reviews_canonical),
        "cross_review_id_merged_groups": int(reviews_canonical["was_cross_review_id_merged"].sum()),
        "same_text_candidate_groups": len(same_text_review_candidates),
        "cross_product_reused_text_groups": len(cross_product_reused_text),
        "cross_product_reused_review_rows": int(reviews_canonical["is_cross_product_reused_text"].sum()),
        "reviews_text_rows": len(reviews_text),
        "product_document_rows": len(product_documents),
    },
    "assertions": {
        check_name: bool(passed)
        for check_name, passed in final_check_report["passed"].items()
    },
    "audit_outputs": {
        "same_text_review_candidates": {
            "path": "data/audit/same_text_review_candidates.csv",
            "row_count": len(same_text_review_candidates),
            "purpose": "Groups with repeated normalized text for one canonical product that were not automatically merged.",
        },
        "cross_product_reused_text": {
            "path": "data/audit/cross_product_reused_text.csv",
            "row_count": len(cross_product_reused_text),
            "purpose": "Normalized review texts observed for more than one canonical product.",
        },
    },
}
quality_report_path = output_dir / "etl_quality_report.json"
quality_report_path.write_text(
    json.dumps(quality_report, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
assert quality_report_path.exists()
display(quality_report)


{'etl_version': '1.0.0',
 'generated_at_utc': '2026-08-18T21:20:57.864878+00:00',
 'stable_metrics': {'raw_product_rows': 1373,
  'raw_review_rows': 314029,
  'product_id_map_rows': 1373,
  'product_variant_rows': 1278,
  'canonical_product_rows': 1268,
  'unmatched_review_rows': 8298,
  'matched_technical_duplicates_removed': 219059,
  'canonical_review_rows': 60368,
  'cross_review_id_merged_groups': 223,
  'same_text_candidate_groups': 24,
  'cross_product_reused_text_groups': 1069,
  'cross_product_reused_review_rows': 2418,
  'reviews_text_rows': 25797,
  'product_document_rows': 1268},
 'assertions': {'product_id_map_rows': True,
  'product_variants_rows': True,
  'product_variants_unique_item_id': True,
  'products_canonical_rows': True,
  'products_canonical_unique': True,
  'products_brand_complete': True,
  'products_category_complete': True,
  'products_description_complete': True,
  'unmatched_review_rows': True,
  'reviews_canonical_rows': True,
  'reviews_canonical_unique